In [ ]:
#imports

import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
# paths and connection

project_root = Path("..")
cleaned_dir = project_root / "data" / "cleaned"
raw_dir = project_root / "data" / "raw"

con = duckdb.connect()

In [ ]:
# choose cost assumptions
fuel_cost_per_mile = 0.30
# later we can do a sensitivity analysis by varying this parameter

In [ ]:
# we want, for each pickup zone and pickup hour:
# average net trip earnings
# average trip duration
# trip count

zone_hour_stats = con.execute(f"""
SELECT
    t.PULocationID,
    EXTRACT(hour FROM tpep_pickup_datetime) AS pickup_hour,
    z.Borough,
    z.Zone,
    COUNT(*) AS trips,

    AVG(
        total_amount - {fuel_cost_per_mile} * trip_distance
    ) AS avg_net_trip_earnings,

    AVG(
        EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 60.0
    ) AS avg_trip_minutes

FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet') t
LEFT JOIN read_csv_auto('{raw_dir.as_posix()}/taxi_zone_lookup.csv') z
    ON t.PULocationID = z.LocationID

WHERE
    total_amount > 0
    AND z.Borough != 'EWR'
    AND EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 8 AND 15

GROUP BY
    t.PULocationID,
    pickup_hour,
    z.Borough,
    z.Zone

HAVING COUNT(*) >= 100
ORDER BY
    pickup_hour,
    trips DESC
""").fetchdf()

zone_hour_stats.head()

In [ ]:
# build hourly transition probabilities
# we want P(Dropoff Zone j | Pickup Zone i, Pickup Hour)
zone_hour_transitions = con.execute(f"""
SELECT
    PULocationID,
    DOLocationID,
    EXTRACT(hour FROM tpep_pickup_datetime) AS pickup_hour,
    COUNT(*) AS trips,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (
        PARTITION BY PULocationID, EXTRACT(hour FROM tpep_pickup_datetime)
    ) AS transition_prob
FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet')
WHERE
    total_amount > 0
    AND EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 8 AND 15
GROUP BY
    PULocationID,
    DOLocationID,
    pickup_hour
""").fetchdf()

zone_hour_transitions.head()

In [ ]:
# keep only zones with enough data

valid_zones = (
    zone_hour_stats.groupby("PULocationID")["trips"]
    .sum()
    .loc[lambda x: x >= 1000]
    .index
)

zone_hour_stats = zone_hour_stats[
    zone_hour_stats["PULocationID"].isin(valid_zones)
].copy()

zone_hour_transitions = zone_hour_transitions[
    zone_hour_transitions["PULocationID"].isin(valid_zones) &
    zone_hour_transitions["DOLocationID"].isin(valid_zones)
].copy()

In [ ]:
# this makes simulation easier since we can just look up stats and transitions
# in memory instead of querying a database

stats_lookup = {}
for _, row in zone_hour_stats.iterrows():
    stats_lookup[(int(row["PULocationID"]), int(row["pickup_hour"]))] = {
        "avg_net_trip_earnings": float(row["avg_net_trip_earnings"]),
        "avg_trip_minutes": float(row["avg_trip_minutes"]),
        "zone_name": row["Zone"],
        "borough": row["Borough"],
    }

# forcing each probability vector to sum to exactly 1
transitions_lookup = {}
for (pu, hr), group in zone_hour_transitions.groupby(["PULocationID", "pickup_hour"]):
    destinations = group["DOLocationID"].astype(int).to_numpy()
    probabilities = group["transition_prob"].to_numpy(dtype=float)

    probabilities = probabilities / probabilities.sum()

    transitions_lookup[(int(pu), int(hr))] = {
        "destinations": destinations,
        "probabilities": probabilities
    }

In [ ]:
# define a function to simulate a shift
def simulate_shift(start_zone, start_hour, shift_minutes=480, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    current_zone = int(start_zone)
    elapsed_minutes = 0.0
    total_net_earnings = 0.0
    trip_count = 0
    path = []

    while elapsed_minutes < shift_minutes:
        current_hour = start_hour + int(elapsed_minutes // 60)

        if current_hour > 15:
            break

        stats_key = (current_zone, current_hour)
        trans_key = (current_zone, current_hour)

        if stats_key not in stats_lookup or trans_key not in transitions_lookup:
            break

        trip_stats = stats_lookup[stats_key]
        total_net_earnings += trip_stats["avg_net_trip_earnings"]
        elapsed_minutes += trip_stats["avg_trip_minutes"]
        trip_count += 1

        path.append({
            "zone": current_zone,
            "hour": current_hour,
            "net_earnings": trip_stats["avg_net_trip_earnings"],
            "trip_minutes": trip_stats["avg_trip_minutes"]
        })

        destinations = transitions_lookup[trans_key]["destinations"]
        probabilities = transitions_lookup[trans_key]["probabilities"]

        current_zone = int(rng.choice(destinations, p=probabilities))

    return {
        "start_zone": start_zone,
        "total_net_earnings": total_net_earnings,
        "hours_worked": elapsed_minutes / 60.0,
        "net_earnings_per_hour": total_net_earnings / (elapsed_minutes / 60.0) if elapsed_minutes > 0 else np.nan,
        "trip_count": trip_count,
        "path": path
    }

In [ ]:
# simulate many shifts for each starting zone and summarize results

def evaluate_start_zone(start_zone, start_hour=8, n_simulations=200):
    results = []
    rng = np.random.default_rng(123)

    for _ in range(n_simulations):
        sim = simulate_shift(
            start_zone=start_zone,
            start_hour=start_hour,
            rng=rng
        )
        results.append(sim["net_earnings_per_hour"])

    return np.nanmean(results), np.nanstd(results)

In [ ]:
# now evaluate all valid starting zones 
zone_results = []

for zone_id in sorted(valid_zones):
    mean_eph, std_eph = evaluate_start_zone(zone_id, n_simulations=200)

    zone_name = zone_hour_stats.loc[
        zone_hour_stats["PULocationID"] == zone_id, "Zone"
    ].iloc[0]

    borough = zone_hour_stats.loc[
        zone_hour_stats["PULocationID"] == zone_id, "Borough"
    ].iloc[0]

    zone_results.append({
        "PULocationID": zone_id,
        "Zone": zone_name,
        "Borough": borough,
        "expected_net_earnings_per_hour": mean_eph,
        "std_net_earnings_per_hour": std_eph
    })

zone_results_df = pd.DataFrame(zone_results).sort_values(
    "expected_net_earnings_per_hour", ascending=False
)

zone_results_df.head(20)

In [ ]:
# plot the top zones as a bar chart

top_zones = zone_results_df.head(15)

plt.figure(figsize=(10,6))

plt.barh(
    top_zones["Zone"],
    top_zones["expected_net_earnings_per_hour"],
    xerr=top_zones["std_net_earnings_per_hour"]
)

plt.gca().invert_yaxis()

plt.xlabel("Expected Net Earnings per Hour")
plt.title("Expected Earnings with Simulation Uncertainty")

plt.show()

In [ ]:
# merge with taxi zone shapefile for mapping

import geopandas as gpd

zones_map = gpd.read_file("../data/raw/taxi_zones/taxi_zones.shp")

map_shift = zones_map.merge(
    zone_results_df,
    left_on="LocationID",
    right_on="PULocationID",
    how="left"
)

In [ ]:
# plot
fig, ax = plt.subplots(1,1, figsize=(12,10))

map_shift.plot(
    column="expected_net_earnings_per_hour",
    cmap="plasma",
    legend=True,
    linewidth=0.5,
    edgecolor="black",
    ax=ax
)

ax.set_title("Expected Net Earnings by Starting Zone\n8-Hour Shift Starting at 12 PM")
ax.axis("off")

plt.show()

In [ ]:
plt.hist([simulate_shift(z)["trip_count"] for z in valid_zones], bins=20)
plt.title("Trips per Simulated Shift")
plt.show()

In [ ]:
sim = simulate_shift(start_zone=236)

[path["zone"] for path in sim["path"]]

In [ ]:
# compare starting at 8am and starting at 12pm for a specific zone

def build_zone_results(start_hour, valid_zones, zone_hour_stats, n_simulations=200):
    zone_results = []

    for zone_id in sorted(valid_zones):
        mean_eph, std_eph = evaluate_start_zone(
            start_zone=zone_id,
            start_hour=start_hour,
            n_simulations=n_simulations
        )

        zone_name = zone_hour_stats.loc[
            zone_hour_stats["PULocationID"] == zone_id, "Zone"
        ].iloc[0]

        borough = zone_hour_stats.loc[
            zone_hour_stats["PULocationID"] == zone_id, "Borough"
        ].iloc[0]

        zone_results.append({
            "PULocationID": zone_id,
            "Zone": zone_name,
            "Borough": borough,
            "expected_net_earnings_per_hour": mean_eph,
            "std_net_earnings_per_hour": std_eph
        })

    return pd.DataFrame(zone_results).sort_values(
        "expected_net_earnings_per_hour", ascending=False
    )

In [ ]:
zone_results_8am = build_zone_results(
    start_hour=8,
    valid_zones=valid_zones,
    zone_hour_stats=zone_hour_stats,
    n_simulations=200
)

zone_results_12pm = build_zone_results(
    start_hour=12,
    valid_zones=valid_zones,
    zone_hour_stats=zone_hour_stats,
    n_simulations=200
)

In [ ]:
zone_results_8am.head()

In [ ]:
zone_results_12pm.head()

In [ ]:
# merge the two result tables

comparison = zone_results_8am.merge(
    zone_results_12pm,
    on=["PULocationID", "Zone", "Borough"],
    suffixes=("_8am", "_12pm")
)

comparison["difference_12pm_minus_8am"] = (
    comparison["expected_net_earnings_per_hour_12pm"]
    - comparison["expected_net_earnings_per_hour_8am"]
)

comparison.head()

In [ ]:
map_compare = zones_map.merge(
    comparison,
    left_on="LocationID",
    right_on="PULocationID",
    how="left"
)

map_compare.head()

In [ ]:
# plot the difference map

fig, ax = plt.subplots(1, 1, figsize=(12,10))

map_compare.plot(
    column="difference_12pm_minus_8am",
    cmap="coolwarm",
    legend=True,
    linewidth=0.5,
    edgecolor="black",
    ax=ax
)

ax.set_title("Difference in Expected Net Earnings per Hour\n12 PM Start minus 8 AM Start", fontsize=14)
ax.axis("off")

plt.show()

In [ ]:
# make a bar chart

top_diff = comparison.sort_values(
    "difference_12pm_minus_8am",
    ascending=False
).head(15)

plt.figure(figsize=(10,6))

plt.barh(
    top_diff["Zone"],
    top_diff["difference_12pm_minus_8am"]
)

plt.gca().invert_yaxis()
plt.xlabel("12 PM minus 8 AM Expected Net Earnings per Hour")
plt.title("Zones That Improve Most When Starting at 12 PM")

plt.show()